# Data audit and label integrity

These cells were executed against the real project outputs. The complete training
and SQL implementation is in the linked source modules. This walkthrough reads and
verifies executed artifacts rather than silently retraining the expensive models.
Home Credit is a consumer-lending proxy; validation is random holdout, never true OOT.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd()
if not (ROOT/'reports/metrics.json').exists():
    ROOT=ROOT.parent
metrics=json.loads((ROOT/'reports/metrics.json').read_text())


## Executed audit

In [2]:
print(json.dumps(metrics['audit'],indent=2))
display(pd.read_csv(ROOT/'reports/missing_values.csv').head(20))
display(pd.read_csv(ROOT/'reports/numeric_audit.csv').head(12))

{
  "rows": 307511,
  "features": 168,
  "bad_count": 24825,
  "bad_rate": 0.08072881945686496,
  "duplicates": 0,
  "employment_sentinels_replaced": 55374,
  "split_design": "Stratified random holdout; no chronology claim. Calibration partition carved from training pool.",
  "splits": [
    {
      "split": "train",
      "rows": 147604,
      "bad_count": 11916,
      "bad_rate": 0.08072951952521612
    },
    {
      "split": "calibration",
      "rows": 36902,
      "bad_count": 2979,
      "bad_rate": 0.08072733185193215
    },
    {
      "split": "test",
      "rows": 61502,
      "bad_count": 4965,
      "bad_rate": 0.08072908198107379
    },
    {
      "split": "holdout",
      "rows": 61503,
      "bad_count": 4965,
      "bad_rate": 0.08072776937710356
    }
  ],
  "income_segments": [
    {
      "segment": "Working",
      "applications": 158774,
      "bads": 15224,
      "bad_rate": 0.09588471664126369,
      "feature": "NAME_INCOME_TYPE",
      "volume_share": 0.516319

,Unnamed: 0,missing_share
0,card_payment_min_ratio,0.807230
1,card_max_utilization,0.720218
2,card_avg_utilization,0.720218
3,card_month_count,0.717392
4,card_avg_dpd,0.717392
5,card_loan_count,0.717392
6,card_delinquent_share,0.717392
7,card_max_dpd,0.717392
8,bureau_delinquent_month_share,0.700674
9,bureau_months_observed,0.700674


,Unnamed: 0,count,mean,std,min,1%,25%,50%,75%,99%,max,iqr_outlier_share
0,SK_ID_CURR,307511.0,278180.518577,102790.175348,100002.00000,103590.200000,189145.500000,278202.00000,367142.500000,4.527129e+05,4.562550e+05,0.000000
1,TARGET,307511.0,0.080729,0.272419,0.00000,0.000000,0.000000,0.00000,0.000000,1.000000e+00,1.000000e+00,0.080729
2,CNT_CHILDREN,307511.0,0.417052,0.722121,0.00000,0.000000,0.000000,0.00000,1.000000,3.000000e+00,1.900000e+01,0.013892
3,AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.00000,45000.000000,112500.000000,147150.00000,202500.000000,4.725000e+05,1.170000e+08,0.045641
4,AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.00000,76410.000000,270000.000000,513531.00000,808650.000000,1.854000e+06,4.050000e+06,0.021339
5,AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.50000,6182.910000,16524.000000,24903.00000,34596.000000,7.000650e+04,2.580255e+05,0.024402
6,AMT_GOODS_PRICE,307233.0,538396.207429,369446.460540,40500.00000,67500.000000,238500.000000,450000.00000,679500.000000,1.800000e+06,4.050000e+06,0.047894
7,REGION_POPULATION_RELATIVE,307511.0,0.020868,0.013831,0.00029,0.002134,0.010006,0.01885,0.028663,7.250800e-02,7.250800e-02,0.027355
8,DAYS_BIRTH,307511.0,-16036.995067,4363.988632,-25229.00000,-24419.000000,-19682.000000,-15750.00000,-12413.000000,-8.263000e+03,-7.489000e+03,0.000000
9,DAYS_EMPLOYED,252137.0,-2384.169325,2338.360162,-17912.00000,-11338.280000,-3175.000000,-1648.00000,-767.000000,-1.110000e+02,0.000000e+00,0.049263


## Interpretation

TARGET records repayment difficulty with provider-withheld overdue/timing thresholds.
Missing histories remain missing. Positive employment-day sentinel values were replaced
with null in SQL; outliers were measured, not silently deleted. See src/audit.py.